In [ ]:
# preparation and initial imports

arguments = True

# mount data and code directory on Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import csv
path = '/content/drive/My Drive/Colab Notebooks/'
sys.path.append(path)

import persuasion_utils as util

!pip install langchain langchain-openai langchain-anthropic langchain-qwen3 langchain-google-genai langchain-deepseek

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import HumanMessage

gemini_api_key = "redacted"
openai_api_key = "redacted"
anthropic_api_key = "redacted"
deepseek_api_key = "redacted"

In [ ]:
# end setup

prompt_labels = [ 'prompt_id', 'argstring', 'prompt' ]

prompts = util.get_judge_prompts( os.path.join( path, 'data', 'judge_prompts_{}.csv'.format( "arguments" if arguments else "noarguments") ) )
total = len( prompts )



In [ ]:
# skip the first k prompts because they've been done before in a session that didn't complete
k = 0

if k > 0:
  prompts = prompts[k:]

to_repeat = []

if to_repeat is not None and len(to_repeat) > 0:
  prompts = [ prompts[ p - 1 ] for p in to_repeat ]



In [ ]:
model_string = "deepseek-reasoner_defaults" # <-- edit this string to invoke different models/model setups

with open( os.path.join( path, model_string.replace("/","-") + "_judgments{}.csv".format( '' if arguments else "_noarguments")), 'a' ) as f:
  writer = csv.writer(f)

  while( len( prompts ) > 0 ):
    ( prompt_id, argstring, prompt_text ) = prompts[0]



    if model_string == "gpt-5-nano_minimal-reasoning":
      llm = ChatOpenAI(
        model=model_string[:model_string.index("_")],
        reasoning_effort="minimal", # "none" is not supported by this model (minimal/low/medium/high only)
        api_key=openai_api_key
      )
    elif model_string == "gpt-5-nano_medium-reasoning":
      llm = ChatOpenAI(
          model=model_string[:model_string.index("_")],
          reasoning_effort="medium", # "none" is not supported by this model (minimal/low/medium/high only)
          api_key=openai_api_key
      )
    elif model_string == "claude-haiku-4-5-20251001_8k-thinking":
      llm = ChatAnthropic(
        model=model_string[:model_string.index("_")],
        thinking={'type' : 'enabled', 'budget_tokens': 8192 },
        max_tokens=16000,
        api_key=anthropic_api_key
      )
    elif model_string == "claude-haiku-4-5-20251001_nothinking":
      llm = ChatAnthropic(
          model=model_string[:model_string.index("_")],
          thinking={'type' : 'disabled' },
          max_tokens=16000,
          api_key=anthropic_api_key
      )
    elif model_string == "gemini-2.5-flash-lite_reasoning":
      llm = ChatGoogleGenerativeAI(
          model=model_string[:model_string.index("_")],
          thinking_budget=-1,
          api_key=gemini_api_key
      )
    elif model_string == "gemini-2.5-flash-lite_nonreasoning":
      llm = ChatGoogleGenerativeAI(
        model=model_string[:model_string.index("_")],
        thinking_budget=0,
        api_key=gemini_api_key
      )
    elif model_string == "gpt-5.1_low-reasoning":
      llm = ChatOpenAI(
          model=model_string[:model_string.index("_")],
          reasoning_effort="low", # "none" is not supported by this model (minimal/low/medium/high only)
          api_key=openai_api_key
      )
    elif model_string == "gpt-5.1_medium-reasoning":
      llm = ChatOpenAI(
          model=model_string[:model_string.index("_")],
          reasoning_effort="medium", # "none" is not supported by this model (minimal/low/medium/high only)
          api_key=openai_api_key
      )
    elif model_string == "claude-sonnet-4-5-20250929_1k-thinking":
      llm = ChatAnthropic(
          model=model_string[:model_string.index("_")],
          thinking={'type' : 'enabled', 'budget_tokens': 1024 },
          max_tokens=16000,
          api_key=anthropic_api_key
      )
    elif model_string == "claude-sonnet-4-5-20250929_8k-thinking":
      llm = ChatAnthropic(
          model=model_string[:model_string.index("_")],
          thinking={'type' : 'enabled', 'budget_tokens': 8192 },
          max_tokens=16000,
          api_key=anthropic_api_key
      )
    elif model_string == "gemini-3-pro-preview_1k-thinking":
      llm = ChatGoogleGenerativeAI(
        model=model_string[:model_string.index("_")],
        thinking_budget=1024,
        api_key=gemini_api_key
      )
    elif model_string == "claude-haiku-4-5-20251001_8k-thinking":
      llm = ChatAnthropic(
          model=model_string[:model_string.index("_")],
          thinking={'type' : 'enabled', 'budget_tokens': 8192 },
          max_tokens=16000,
          api_key=anthropic_api_key
      )
    elif model_string == "gemini-3-pro-preview_8k-thinking":
      llm = ChatGoogleGenerativeAI(
        model=model_string[:model_string.index("_")],
        thinking_budget=8192,
        api_key=gemini_api_key
      )
    elif model_string == "deepseek-chat_defaults":
      llm = ChatDeepSeek(
          model=model_string[:model_string.index("_")],
          api_key=deepseek_api_key
      )
    elif model_string == "deepseek-reasoner_defaults":
      llm = ChatDeepSeek(
          model=model_string[:model_string.index("_")],
          api_key=deepseek_api_key
      )

    if model_string.startswith( "deepseek" ):
      response = llm.invoke([HumanMessage(content=prompt_text)])
    else:
      response = llm.invoke(prompt_text)

    writer.writerow( [ prompt_id, argstring, model_string, response ] )
    print( "Finished {} of {}".format( prompt_id, total ) )
    prompts.pop(0) #only remove from the list once it's completed successfully
print( "Job complete")
